In [1]:
import pandas as pd
import os

In [13]:
df_medium_valid = pd.read_csv("boxoban-astar-solutions/medium_valid.csv")
filtered = df_medium_valid[
    df_medium_valid["Steps"] != "INCORRECT_SOLUTION_FOUND"
].copy()
filtered = filtered[filtered["Actions"] != "SEARCH_STATE_FAILED"].copy()


len(filtered)
filtered

,File,Level,Actions,Steps,SearchSteps
0,0,0,3230233000330001232211222330010122033000100321...,65,119046
1,0,1,32330321221011300332330012,26,10580
2,0,2,1121211003212322100303321012,28,3956
3,0,3,12321212230003001212232212230,29,1855
4,0,4,003321200111121103333211012203333233011111012,45,80334
...,...,...,...,...,...
49995,49,995,3001112232100033322213000002211122332300002111...,56,12540
49996,49,996,2212333302111003232133331011121,31,848
49997,49,997,3332223303300123221111000100333322102332221100...,53,29641
49998,49,998,2233330000110112223212310000033213332212112103...,57,31207


In [3]:
def load_level_by_id(path: str, level_id: str) -> str:
    """
    path: path to the .txt level file
    level_id: stringified triple, e.g. "000", "014"

    Returns:
        Level grid as a single string (with newlines)
    """
    target = int(level_id)  # "000" -> 0
    current_id = None
    collecting = False
    level_lines = []

    with open(path, "r") as f:
        for line in f:
            line = line.rstrip("\n")

            # Level header
            if line.startswith(";"):
                # Stop if we were collecting and hit next level
                if collecting:
                    break

                # Parse level number
                try:
                    current_id = int(line[1:].strip())
                except ValueError:
                    current_id = None

                collecting = (current_id == target)
                continue

            # Collect level lines
            if collecting:
                level_lines.append(line)

    if not level_lines:
        raise ValueError(f"Level {level_id} not found in file")

    return "\n".join(level_lines)

level = load_level_by_id("boxoban-levels/medium/valid/000.txt", "000")
print(level)


##########
#       ##
# $    ###
#. ## . ##
# ####  ##
#   #   ##
#$$ #. ###
# # # $@##
# .    ###
##########



In [4]:
from typing import Set, Tuple

def parse_sokoban_level(level_str: str):
    """
    Parses a 10x10 Sokoban level string.

    Returns:
        walls  : Set[(r, c)]
        boxes  : Set[(r, c)]
        goals  : Set[(r, c)]
        player : (r, c)
    """
    walls: Set[Tuple[int, int]] = set()
    boxes: Set[Tuple[int, int]] = set()
    goals: Set[Tuple[int, int]] = set()
    player = None

    rows = level_str.split("\n")

    for r, row in enumerate(rows):
        for c, ch in enumerate(row):
            if ch == "#":
                walls.add((r, c))

            elif ch == "$":
                boxes.add((r, c))

            elif ch == ".":
                goals.add((r, c))

            elif ch == "@":
                player = (r, c)

            elif ch == "*":          # box on goal
                boxes.add((r, c))
                goals.add((r, c))

            elif ch == "+":          # player on goal
                player = (r, c)
                goals.add((r, c))

    if player is None:
        raise ValueError("No player found in level")

    return walls, boxes, goals, player

In [8]:
state = parse_sokoban_level(level)
action_map = [(-1,0),(0,1),(1,0),(0,-1)]

actions = filtered.head(1)["Actions"].item()
states = [state]
for action in actions:
    walls, boxes, goals, player = state
    if tuple(sorted(boxes)) == tuple(sorted(goals)):
        print("solved")
        break
    dy, dx = action_map[int(action)]
    new_pos_player = (player[0]+dy, player[1]+dx)
    if new_pos_player in walls:
        continue
    if new_pos_player in boxes:
        new_pos_box = (new_pos_player[0]+dy, new_pos_player[1]+dx)
        if new_pos_box in boxes:
            continue
        boxes.remove(new_pos_player)
        boxes.add(new_pos_box)
    player = new_pos_player

    state = (walls, boxes, goals, player)
    states.append(state)

if tuple(sorted(boxes)) == tuple(sorted(goals)):
    print("solved")

solved


In [12]:
state[0]

{(0, 0),
 (0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (0, 5),
 (0, 6),
 (0, 7),
 (0, 8),
 (0, 9),
 (1, 0),
 (1, 8),
 (1, 9),
 (2, 0),
 (2, 7),
 (2, 8),
 (2, 9),
 (3, 0),
 (3, 3),
 (3, 4),
 (3, 8),
 (3, 9),
 (4, 0),
 (4, 2),
 (4, 3),
 (4, 4),
 (4, 5),
 (4, 8),
 (4, 9),
 (5, 0),
 (5, 4),
 (5, 8),
 (5, 9),
 (6, 0),
 (6, 4),
 (6, 7),
 (6, 8),
 (6, 9),
 (7, 0),
 (7, 2),
 (7, 4),
 (7, 8),
 (7, 9),
 (8, 0),
 (8, 7),
 (8, 8),
 (8, 9),
 (9, 0),
 (9, 1),
 (9, 2),
 (9, 3),
 (9, 4),
 (9, 5),
 (9, 6),
 (9, 7),
 (9, 8),
 (9, 9)}

In [ ]:
my_str = str(filtered.head(1)["File"].item())
while len(my_str) < 3:
    my_str = "0" + my_str
print(my_str)

000


In [7]:
actions

'32302330003300012322112223300101220330001003211110122122302223233'

In [10]:
states[0]

({(0, 0),
  (0, 1),
  (0, 2),
  (0, 3),
  (0, 4),
  (0, 5),
  (0, 6),
  (0, 7),
  (0, 8),
  (0, 9),
  (1, 0),
  (1, 8),
  (1, 9),
  (2, 0),
  (2, 7),
  (2, 8),
  (2, 9),
  (3, 0),
  (3, 3),
  (3, 4),
  (3, 8),
  (3, 9),
  (4, 0),
  (4, 2),
  (4, 3),
  (4, 4),
  (4, 5),
  (4, 8),
  (4, 9),
  (5, 0),
  (5, 4),
  (5, 8),
  (5, 9),
  (6, 0),
  (6, 4),
  (6, 7),
  (6, 8),
  (6, 9),
  (7, 0),
  (7, 2),
  (7, 4),
  (7, 8),
  (7, 9),
  (8, 0),
  (8, 7),
  (8, 8),
  (8, 9),
  (9, 0),
  (9, 1),
  (9, 2),
  (9, 3),
  (9, 4),
  (9, 5),
  (9, 6),
  (9, 7),
  (9, 8),
  (9, 9)},
 {(3, 1), (3, 6), (6, 5), (8, 2)},
 {(3, 1), (3, 6), (6, 5), (8, 2)},
 (7, 7))